In [3]:
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [8]:
import pandas as pd

df = pd.read_parquet("../data/processed/arxiv_ml.parquet")

In [9]:
df.head()

,id,title,authors,category,text_raw,text_ml
0,704.0001,Calculation of prompt diphoton production cros...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",hep-ph,Calculation of prompt diphoton production cros...,calculation prompt diphoton production cross s...
1,704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,math.CO,Sparsity-certifying Graph Decompositions We ...,sparsity certify graph decomposition describe ...
2,704.0003,The evolution of the Earth-Moon system based o...,Hongjun Pan,physics.gen-ph,The evolution of the Earth-Moon system based o...,evolution earth moon system base dark matter f...
3,704.0004,A determinant of Stirling cycle numbers counts...,David Callan,math.CO,A determinant of Stirling cycle numbers counts...,determinant stirling cycle number count unlabe...
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,Wael Abu-Shammala and Alberto Torchinsky,math.CA,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,dyadic lambda alpha lambda alpha compute lambd...


In [10]:
print(df.shape)

print(df["category"].nunique())

print(df["text_ml"].isnull().sum())

print(df["category"].isnull().sum())

(1000, 6)
87
0
0


In [11]:
X = df["text_ml"]

y = df["category"]

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

ValueError: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: ['cs.CE', 'cs.DM', 'cs.NI', 'math.KT', 'math.LO', 'math.MG', 'math.SP', 'nlin.CG', 'physics.acc-ph', 'physics.atm-clus', 'physics.class-ph', 'physics.geo-ph', 'physics.pop-ph', 'q-bio.QM', 'q-bio.SC', 'q-fin.CP', 'q-fin.GN', 'q-fin.PR', 'q-fin.RM', 'stat.ME']

In [13]:
category_counts = df["category"].value_counts()

category_counts.tail(30)

category
q-bio.OT            2
physics.plasm-ph    2
physics.bio-ph      2
physics.flu-dyn     2
q-bio.NC            2
q-bio.BM            2
cs.CR               2
q-fin.ST            2
math.GM             2
physics.hist-ph     2
q-bio.QM            1
physics.pop-ph      1
cs.CE               1
nlin.CG             1
cs.DM               1
math.MG             1
physics.geo-ph      1
q-fin.RM            1
q-bio.SC            1
math.LO             1
cs.NI               1
q-fin.PR            1
physics.class-ph    1
q-fin.GN            1
stat.ME             1
q-fin.CP            1
math.KT             1
physics.atm-clus    1
physics.acc-ph      1
math.SP             1
Name: count, dtype: int64

In [14]:
valid_categories = category_counts[category_counts >= 2].index

df_ml = df[df["category"].isin(valid_categories)].copy()

In [15]:
X = df_ml["text_ml"]

y = df_ml["category"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
category_counts = df["category"].value_counts()

rare_categories = category_counts[category_counts < 2]

print(f"Rare Categories : {len(rare_categories)}")
print(f"Papers Removed  : {rare_categories.sum()}")

rare_categories

Rare Categories : 20
Papers Removed  : 20


category
q-bio.QM            1
physics.pop-ph      1
cs.CE               1
nlin.CG             1
cs.DM               1
math.MG             1
physics.geo-ph      1
q-fin.RM            1
q-bio.SC            1
math.LO             1
cs.NI               1
q-fin.PR            1
physics.class-ph    1
q-fin.GN            1
stat.ME             1
q-fin.CP            1
math.KT             1
physics.atm-clus    1
physics.acc-ph      1
math.SP             1
Name: count, dtype: int64

In [18]:
category_counts = df["category"].value_counts()

valid_categories = category_counts[category_counts >= 2].index

df_ml = df[df["category"].isin(valid_categories)].copy()

print(df_ml.shape)
print(df_ml["category"].nunique())

(980, 6)
67


In [19]:
X = df_ml["text_ml"]
y = df_ml["category"]

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(784,)
(196,)
(784,)
(196,)


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [23]:
print("Train Matrix :", X_train_tfidf.shape)
print("Test Matrix  :", X_test_tfidf.shape)

print("Vocabulary Size :", len(tfidf.get_feature_names_out()))

Train Matrix : (784, 5000)
Test Matrix  : (196, 5000)
Vocabulary Size : 5000


In [24]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:30])

['10' '12' '1d' '1e' '1t' '20' '24um' '2cu' '2d' '2dfgrs' '2e' '2mass'
 '2n' '2s' '2s meson' '30' '3c' '3d' '43' '4d' '4n' '61' 'a1' 'aa' 'ab'
 'ab initio' 'abelian' 'ability' 'able' 'abridge']


In [25]:
print(feature_names[-30:])

['wonderful' 'work' 'world' 'worldsheet' 'x10' 'xcuo' 'xi' 'xmm'
 'xmm newton' 'xrt' 'xxz' 'yang' 'yang mill' 'yang mills' 'yau' 'year'
 'year ago' 'yield' 'yield good' 'yield ratio' 'york' 'young' 'young star'
 'young stellar' 'yr' 'yso' 'zds' 'zero' 'zn' 'zone']


In [26]:
important_words = [
    "quantum",
    "graph",
    "neural",
    "galaxy",
    "energy",
    "black hole",
    "machine learning"
]

for word in important_words:
    if word in feature_names:
        print(f"✅ {word}")
    else:
        print(f"❌ {word}")

✅ quantum
✅ graph
✅ neural
✅ galaxy
✅ energy
✅ black hole
❌ machine learning


In [27]:
from sklearn.svm import LinearSVC

svm = LinearSVC(
    random_state=42,
    dual="auto"
)

svm.fit(X_train_tfidf, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adj

In [28]:
y_pred = svm.predict(X_test_tfidf)

In [29]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")

Accuracy : 0.5663


In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

          astro-ph       0.84      0.97      0.90        38
   cond-mat.dis-nn       0.00      0.00      0.00         2
 cond-mat.mes-hall       0.50      0.50      0.50         8
 cond-mat.mtrl-sci       0.75      0.50      0.60         6
    cond-mat.other       0.00      0.00      0.00         4
     cond-mat.soft       0.00      0.00      0.00         3
cond-mat.stat-mech       0.50      0.20      0.29         5
   cond-mat.str-el       0.36      0.62      0.45         8
 cond-mat.supr-con       0.40      0.40      0.40         5
             cs.CC       0.00      0.00      0.00         1
             cs.DS       0.00      0.00      0.00         1
             cs.IT       0.67      0.67      0.67         3
             cs.NE       0.50      1.00      0.67         1
             cs.PF       0.00      0.00      0.00         1
             gr-qc       0.83      0.71      0.77         7
            hep-ex       1.00      0.33

c:\Users\DELL\Desktop\Cdac\aiml-project\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Desktop\Cdac\aiml-project\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Desktop\Cdac\aiml-project\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz

In [31]:
pd.Series(y_pred).value_counts().head(20)

astro-ph             44
hep-th               18
hep-ph               16
cond-mat.str-el      14
quant-ph             13
cond-mat.mes-hall     8
math.CO               6
physics.optics        6
gr-qc                 6
cond-mat.supr-con     5
nucl-ex               4
math.AG               4
math.DG               4
cond-mat.mtrl-sci     4
math.PR               4
cond-mat.other        3
math.GR               3
cs.IT                 3
cond-mat.soft         2
cs.NE                 2
Name: count, dtype: int64

In [32]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Text": X_test.values
})

wrong = results[
    results["Actual"] != results["Predicted"]
]

wrong.head(10)

,Actual,Predicted,Text
4,physics.gen-ph,hep-th,universe singularity group de sitter cosmology...
7,math.DS,math.GT,dynamical object cohomologically expand map go...
10,cond-mat.mes-hall,hep-th,spin accumulation non abelian aharonov bohm ef...
12,cond-mat.mes-hall,cond-mat.str-el,thermal entanglement qubit pair shastry suther...
13,cond-mat.stat-mech,hep-th,aspect nonperturbative renormalization phi mod...
14,cond-mat.stat-mech,cond-mat.str-el,density matrix element entanglement entropy sp...
15,nucl-th,nucl-ex,scale nucleus nuclear matter know correlation ...
16,astro-ph,quant-ph,measurement aerosol phase function pierre auge...
17,math-ph,cond-mat.other,dynamic bose einstein condensate report recent...
18,math.CO,math.AG,algorithm classification smooth fano polytope ...


In [33]:
feature_names = tfidf.get_feature_names_out()

In [34]:
for i, category in enumerate(svm.classes_[:5]):

    top = svm.coef_[i].toarray().flatten().argsort()[-15:]

    print(category)

    print(feature_names[top])

    print()

AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [35]:
top = svm.coef_[i].argsort()[-15:]

In [36]:
feature_names = np.array(tfidf.get_feature_names_out())

for i, category in enumerate(svm.classes_[:10]):

    top = np.argsort(svm.coef_[i])[-15:]

    print("="*80)
    print(category)
    print("="*80)

    print(feature_names[top])

astro-ph
['solar' 'emission' 'dwarf' 'survey' 'galactic' 'cluster' 'disk'
 'velocity' 'mass' 'source' 'observation' 'planet' 'ray' 'galaxy' 'star']
cond-mat.dis-nn
['glassy system' 'ee' 'decagonal' 'simulation' 'ring'
 'entanglement entropy' 'metallic' 'growth' 'electron electron'
 'electron interaction' '3d' 'mathrm' 'ising' 'ising model' 'upsilon']
cond-mat.mes-hall
['wire' 'enhance' 'impurity' 'boson' 'hall' 'current' 'nanotube'
 'persistent' 'plasmon' 'wave function' 'graphene' 'conductance' 'contact'
 'electron' 'dot']
cond-mat.mtrl-sci
['metal' 'density functional' 'efficiency' 'edge' 'domain wall'
 'nanoparticle' 'layer' 'zn' 'dna' 'swnt' 'step' 'structural' 'gpa'
 'melting' 'film']
cond-mat.other
['transistor' 'ps' 'emit' 'bose' 'bose einstein' 'spin'
 'einstein condensate' 'soliton' 'condensate' 'organic' 'dipole'
 'light emit' 'random' 'ultracold' 'oscillation']
cond-mat.soft
['poly' 'concentration' 'dynamic' 'ion' 'water' 'network' 'acid'
 'percolation' 'crossover' 'knot' 'c

In [37]:
feature_names = np.array(tfidf.get_feature_names_out())

top_features = []

for i, category in enumerate(svm.classes_):

    top = np.argsort(svm.coef_[i])[-15:]

    words = feature_names[top]

    for word in words:
        top_features.append([category, word])

top_features_df = pd.DataFrame(
    top_features,
    columns=["Category", "Important Word"]
)

top_features_df.head(20)

,Category,Important Word
0,astro-ph,solar
1,astro-ph,emission
2,astro-ph,dwarf
3,astro-ph,survey
4,astro-ph,galactic
5,astro-ph,cluster
6,astro-ph,disk
7,astro-ph,velocity
8,astro-ph,mass
9,astro-ph,source
